# Séance 7 – Classes et instances (2)
### Lab : attributs de classe, méthodes spéciales et propriétés – **CORRIGÉ (version formateur)**

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 5 : Equipment complète, Employee et Inventory |

**Objectifs**
- Utiliser attributs de classe, @classmethod et @staticmethod
- Implémenter les méthodes spéciales usuelles
- Encapsuler avec _, __ et @property

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Attributs de classe, méthodes de classe et statiques

> **Rappel**
> - Attribut de classe : défini dans le corps de la classe, partagé ; modifié via `NomClasse.attr`
> - `@classmethod def f(cls, ...)` : reçoit la classe → constructeurs alternatifs `return cls(...)`
> - `@staticmethod def f(...)` : fonction utilitaire sans `self` ni `cls`

### Exemple guidé : compteur d'instances et constructeur alternatif

In [1]:
class Employee:
    """Employé de l'entreprise."""

    DOMAIN = "entreprise.tn"
    headcount = 0

    def __init__(self, first_name: str, last_name: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        Employee.headcount += 1

    @classmethod
    def from_email(cls, email: str) -> "Employee":
        local = email.split("@")[0]
        first, last = local.split(".")
        return cls(first.capitalize(), last.capitalize())

    @staticmethod
    def is_professional(email: str) -> bool:
        return email.endswith("@" + Employee.DOMAIN)


karim = Employee.from_email("karim.trabelsi@entreprise.tn")
print(karim.first_name, karim.last_name, Employee.headcount)
print(Employee.is_professional("karim@gmail.com"))

Karim Trabelsi 1
False


### Exercice 1 ⭐ – Numérotation automatique des tickets

Écrivez la classe `Ticket(title: str)` :
- un attribut de classe `_next_number` (initialement `1`) ;
- chaque nouveau ticket reçoit un attribut `ticket_id` de la forme `"INC-0001"`, `"INC-0002"`… ;
- une méthode de classe `reset_counter(cls, start: int = 1)` remet le compteur à `start`.

In [2]:
class Ticket:
    """Ticket numéroté automatiquement."""

    _next_number = 1

    def __init__(self, title: str) -> None:
        self.title = title
        self.ticket_id = f"INC-{Ticket._next_number:04d}"
        Ticket._next_number += 1

    @classmethod
    def reset_counter(cls, start: int = 1) -> None:
        """Réinitialise la numérotation."""
        cls._next_number = start


t1, t2 = Ticket("VPN lent"), Ticket("Écran noir")
print(t1.ticket_id, t2.ticket_id)

INC-0001 INC-0002


<details><summary>💡 Indice</summary>

Dans `__init__`, lisez et incrémentez `Ticket._next_number` (et non `self._next_number += 1`, qui créerait un attribut d'instance).

</details>

In [3]:
# ✅ Vérification
Ticket.reset_counter()
a, b, c = Ticket("A"), Ticket("B"), Ticket("C")
assert (a.ticket_id, b.ticket_id, c.ticket_id) == ("INC-0001", "INC-0002", "INC-0003")
Ticket.reset_counter(500)
assert Ticket("D").ticket_id == "INC-0500"
assert "_next_number" not in vars(a), "le compteur doit rester un attribut de classe"
print("✅ Exercice 1 réussi")

✅ Exercice 1 réussi


### Exercice 2 ⭐⭐ – Corriger un attribut de classe partagé

La classe `Playbook` ci-dessous a deux bugs : les tâches et les hôtes sont partagés entre tous les playbooks.
Corrigez-la **sans changer son interface** (mêmes méthodes, mêmes attributs).

In [4]:
class Playbook:
    """Playbook d'automatisation (liste de tâches appliquées à des hôtes)."""

    DEFAULT_FORKS = 5

    def __init__(self, name: str) -> None:
        self.name = name
        self.tasks: list[str] = []
        self.hosts: set[str] = set()

    def add_task(self, task: str) -> None:
        self.tasks.append(task)

    def target(self, *hosts: str) -> None:
        self.hosts.update(hosts)


web = Playbook("web")
web.add_task("installer nginx")
web.target("srv-web-01", "srv-web-02")
db = Playbook("db")
print(web.tasks, db.tasks, db.hosts, Playbook.DEFAULT_FORKS)

['installer nginx'] [] set() 5


<details><summary>💡 Indice</summary>

`DEFAULT_FORKS` est une constante : elle peut rester attribut de classe. Les collections propres à chaque playbook se créent dans `__init__`.

</details>

In [5]:
# ✅ Vérification
p1, p2 = Playbook("p1"), Playbook("p2")
p1.add_task("t1")
p1.target("h1")
assert p2.tasks == [] and p2.hosts == set()
assert p1.tasks == ["t1"] and p1.hosts == {"h1"}
assert Playbook.DEFAULT_FORKS == 5
print("✅ Exercice 2 réussi")

✅ Exercice 2 réussi


### Exercice 3 ⭐⭐ – Constructeur alternatif et validation

Complétez la classe `Server` :
- `is_valid_ip(ip)` (**statique**) : 4 nombres de 0 à 255 séparés par des points ;
- `from_inventory_line(line)` (**de classe**) : crée un serveur à partir de `"hostname;ip;cpu;ram_gb"` (espaces et retour à la ligne tolérés) ;
- `from_dict(data)` (**de classe**) : à partir d'un dictionnaire avec les clés `hostname`, `ip`, `cpu`, `ram_gb` ;
- l'attribut `valid` (booléen) indique si l'IP est valide (calculé dans `__init__` avec `is_valid_ip`).

In [6]:
class Server:
    """Serveur du parc."""

    def __init__(self, hostname: str, ip: str, cpu: int, ram_gb: int) -> None:
        self.hostname = hostname
        self.ip = ip
        self.cpu = cpu
        self.ram_gb = ram_gb
        self.valid = Server.is_valid_ip(ip)

    @staticmethod
    def is_valid_ip(ip: str) -> bool:
        """Vérifie le format d'une adresse IPv4."""
        parts = ip.split(".")
        if len(parts) != 4:
            return False
        for part in parts:
            if not part.isdigit() or not 0 <= int(part) <= 255:
                return False
        return True

    @classmethod
    def from_inventory_line(cls, line: str) -> "Server":
        """Crée un serveur depuis une ligne 'hostname;ip;cpu;ram_gb'."""
        hostname, ip, cpu, ram = [field.strip() for field in line.strip().split(";")]
        return cls(hostname, ip, int(cpu), int(ram))

    @classmethod
    def from_dict(cls, data: dict) -> "Server":
        """Crée un serveur depuis un dictionnaire."""
        return cls(data["hostname"], data["ip"], data["cpu"], data["ram_gb"])


lines = ["srv-web-01; 10.0.0.5 ;8;32\n", "srv-db-01;10.0.0.300;16;128\n"]
servers = [Server.from_inventory_line(line) for line in lines]
print([(s.hostname, s.ip, s.valid) for s in servers])

[('srv-web-01', '10.0.0.5', True), ('srv-db-01', '10.0.0.300', False)]


<details><summary>💡 Indice</summary>

`[f.strip() for f in line.split(";")]` nettoie chaque champ. Une méthode de classe se termine par `return cls(...)`.

</details>

In [7]:
# ✅ Vérification
assert Server.is_valid_ip("192.168.1.10") and not Server.is_valid_ip("10.0.0") and not Server.is_valid_ip("10.0.0.x")
s = Server.from_inventory_line("srv-app-01 ; 10.0.1.7 ; 4 ; 16\n")
assert (s.hostname, s.ip, s.cpu, s.ram_gb, s.valid) == ("srv-app-01", "10.0.1.7", 4, 16, True)
d = Server.from_dict({"hostname": "srv-x", "ip": "300.1.1.1", "cpu": 2, "ram_gb": 4})
assert isinstance(d, Server) and d.valid is False
print("✅ Exercice 3 réussi")

✅ Exercice 3 réussi


### Exercice 4 ⭐⭐⭐ – La classe Duration

Écrivez la classe `Duration`, qui stocke une durée en **minutes** (attribut `minutes`) :
- `from_hours(hours: float)` : constructeur alternatif ;
- `parse(text: str)` : statique, convertit `"1h30"`, `"45min"`, `"2h"` en nombre de minutes (`int`) ;
- `from_text(text: str)` : constructeur alternatif qui utilise `parse` ;
- `to_text()` : `"1h30"`, `"45min"`, `"2h"` (format inverse ; `"0min"` pour une durée nulle).

In [8]:
class Duration:
    """Durée exprimée en minutes."""

    def __init__(self, minutes: int) -> None:
        self.minutes = minutes

    @classmethod
    def from_hours(cls, hours: float) -> "Duration":
        return cls(round(hours * 60))

    @staticmethod
    def parse(text: str) -> int:
        """Convertit '1h30', '45min' ou '2h' en minutes."""
        text = text.strip().lower()
        if text.endswith("min"):
            return int(text[:-3])
        hours, _, minutes = text.partition("h")
        return int(hours) * 60 + (int(minutes) if minutes else 0)

    @classmethod
    def from_text(cls, text: str) -> "Duration":
        return cls(Duration.parse(text))

    def to_text(self) -> str:
        hours, minutes = divmod(self.minutes, 60)
        if hours == 0:
            return f"{minutes}min"
        return f"{hours}h{minutes:02d}" if minutes else f"{hours}h"


print(Duration.from_text("1h30").minutes, Duration.from_hours(2.25).to_text(), Duration(45).to_text())

90 2h15 45min


<details><summary>💡 Indice</summary>

`"1h30".partition("h")` renvoie `('1', 'h', '30')`. `divmod(95, 60)` renvoie `(1, 35)`.

</details>

In [9]:
# ✅ Vérification
assert Duration.parse("1h30") == 90 and Duration.parse("45min") == 45 and Duration.parse(" 2H ") == 120
assert Duration.from_text("1h05").minutes == 65
assert Duration.from_hours(2.25).minutes == 135
assert Duration(90).to_text() == "1h30" and Duration(120).to_text() == "2h"
assert Duration(45).to_text() == "45min" and Duration(0).to_text() == "0min" and Duration(65).to_text() == "1h05"
print("✅ Exercice 4 réussi")

✅ Exercice 4 réussi


---
## Partie 2 – Méthodes spéciales et encapsulation

> **Rappel**
> - `__str__` (lisible), `__repr__` (non ambigu) · `__eq__`, `__lt__` (comparaison, tri)
> - `__len__`, `__contains__`, `__getitem__` (collections) · `__add__`, `__sub__`, `__mul__` (opérateurs)
> - `_attribut` : interne par convention · `__attribut` : *name mangling*
> - `@property` (lecture) + `@nom.setter` (écriture contrôlée) ; le setter écrit dans `self._nom`

### Exemple guidé : une file d'attente de tickets

In [10]:
class TicketQueue:
    """File d'attente de tickets, utilisable comme une collection."""

    def __init__(self, name: str) -> None:
        self.name = name
        self._tickets: list[str] = []

    def push(self, ticket: str) -> None:
        self._tickets.append(ticket)

    def __len__(self) -> int:
        return len(self._tickets)

    def __contains__(self, ticket: str) -> bool:
        return ticket in self._tickets

    def __getitem__(self, index: int) -> str:
        return self._tickets[index]

    def __str__(self) -> str:
        return f"File {self.name} : {len(self)} ticket(s)"


queue = TicketQueue("Support N1")
queue.push("INC-0001")
queue.push("INC-0002")
print(queue, "INC-0002" in queue, queue[0], queue[-1])
for ticket in queue:           # __getitem__ suffit pour parcourir l'objet
    print("-", ticket)

File Support N1 : 2 ticket(s) True INC-0001 INC-0002
- INC-0001
- INC-0002


### Exercice 5 ⭐ – __str__ et __repr__

Ajoutez à la classe `VirtualMachine` :
- `__str__` → `"vm-web-01 (2 vCPU, 4 Go) – running"` ;
- `__repr__` → `"VirtualMachine('vm-web-01', cpu=2, ram_gb=4)"`.

In [11]:
class VirtualMachine:
    """Machine virtuelle."""

    def __init__(self, name: str, cpu: int = 2, ram_gb: int = 4) -> None:
        self.name = name
        self.cpu = cpu
        self.ram_gb = ram_gb
        self.state = "running"

    def __str__(self) -> str:
        return f"{self.name} ({self.cpu} vCPU, {self.ram_gb} Go) – {self.state}"

    def __repr__(self) -> str:
        return f"VirtualMachine({self.name!r}, cpu={self.cpu}, ram_gb={self.ram_gb})"


vm = VirtualMachine("vm-web-01")
print(vm)
print([vm])

vm-web-01 (2 vCPU, 4 Go) – running
[VirtualMachine('vm-web-01', cpu=2, ram_gb=4)]


<details><summary>💡 Indice</summary>

`{self.name!r}` ajoute les guillemets autour du nom.

</details>

In [12]:
# ✅ Vérification
vm = VirtualMachine("vm-db-01", cpu=8, ram_gb=32)
assert str(vm) == "vm-db-01 (8 vCPU, 32 Go) – running"
assert repr(vm) == "VirtualMachine('vm-db-01', cpu=8, ram_gb=32)"
assert f"{vm}" == str(vm)
print("✅ Exercice 5 réussi")

✅ Exercice 5 réussi


### Exercice 6 ⭐⭐ – Trier des versions de logiciels

Écrivez la classe `SoftwareVersion(text: str)` (`"3.12.1"`, `"2.4"`) :
- attribut `parts` : tuple d'entiers ;
- `__eq__` : deux versions sont égales si leurs `parts` le sont (`"2.4"` est égale à `"2.4.0"` : complétez par des zéros jusqu'à 3 nombres) ;
- `__lt__` : ordre numérique ;
- `__str__` : le texte d'origine ; `__repr__` : `SoftwareVersion('3.12.1')`.

Puis calculez `outdated` : les serveurs dont la version d'OpenSSL est **strictement inférieure** à `minimum`.

In [13]:
class SoftwareVersion:
    """Numéro de version comparable."""

    def __init__(self, text: str) -> None:
        self.text = text
        numbers = [int(p) for p in text.split(".")]
        while len(numbers) < 3:
            numbers.append(0)
        self.parts = tuple(numbers)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, SoftwareVersion) and self.parts == other.parts

    def __lt__(self, other: "SoftwareVersion") -> bool:
        return self.parts < other.parts

    def __str__(self) -> str:
        return self.text

    def __repr__(self) -> str:
        return f"SoftwareVersion({self.text!r})"


openssl = {"srv-web-01": "3.0.13", "srv-web-02": "3.2.1", "srv-db-01": "1.1.1", "srv-app-01": "3.0.2"}
minimum = SoftwareVersion("3.0.13")
outdated = sorted([host for host, v in openssl.items() if SoftwareVersion(v) < minimum])
print(outdated, sorted(SoftwareVersion(v) for v in openssl.values()))

['srv-app-01', 'srv-db-01'] [SoftwareVersion('1.1.1'), SoftwareVersion('3.0.2'), SoftwareVersion('3.0.13'), SoftwareVersion('3.2.1')]


<details><summary>💡 Indice</summary>

`sorted()` n'a besoin que de `__lt__`. Comparer des chaînes donnerait `"3.10" < "3.9"` : faux numériquement !

</details>

In [14]:
# ✅ Vérification
assert SoftwareVersion("2.4") == SoftwareVersion("2.4.0") and SoftwareVersion("3.9") != SoftwareVersion("3.10")
assert SoftwareVersion("3.9.18") < SoftwareVersion("3.12.1")
assert SoftwareVersion("1.0") != "1.0"
assert [str(v) for v in sorted([SoftwareVersion("3.13"), SoftwareVersion("3.9.1"), SoftwareVersion("3.12.0")])] == ["3.9.1", "3.12.0", "3.13"]
assert repr(SoftwareVersion("3.12.1")) == "SoftwareVersion('3.12.1')"
assert outdated == ["srv-app-01", "srv-db-01"]
print("✅ Exercice 6 réussi")

✅ Exercice 6 réussi


### Exercice 7 ⭐⭐ – La classe Money

Écrivez la classe `Money(millimes: int)` (1 DT = 1 000 millimes) :
- `from_dinars(amount: float)` (de classe) : `Money.from_dinars(12.5)` → 12 500 millimes ;
- `__add__` et `__sub__` entre deux `Money` ; `__mul__` par un entier (quantité) ;
- `__eq__`, `__lt__` ;
- `__str__` → `"12.500 DT"` ; `__repr__` → `"Money(12500)"`.

Calculez ensuite `total` (le prix total de la commande) et `remaining` (budget restant).

In [15]:
class Money:
    """Montant en dinars tunisiens, stocké en millimes."""

    def __init__(self, millimes: int) -> None:
        self.millimes = millimes

    @classmethod
    def from_dinars(cls, amount: float) -> "Money":
        return cls(round(amount * 1000))

    def __add__(self, other: "Money") -> "Money":
        return Money(self.millimes + other.millimes)

    def __sub__(self, other: "Money") -> "Money":
        return Money(self.millimes - other.millimes)

    def __mul__(self, quantity: int) -> "Money":
        return Money(self.millimes * quantity)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Money) and self.millimes == other.millimes

    def __lt__(self, other: "Money") -> bool:
        return self.millimes < other.millimes

    def __str__(self) -> str:
        return f"{self.millimes / 1000:.3f} DT"

    def __repr__(self) -> str:
        return f"Money({self.millimes})"


order = [("Clavier", Money.from_dinars(45.9), 10), ("Écran 27 pouces", Money.from_dinars(649.0), 4), ("Câble HDMI", Money(12_350), 6)]
total = Money(0)
for name, unit_price, quantity in order:
    total = total + unit_price * quantity
budget = Money.from_dinars(3500)
remaining = budget - total
print(total, remaining, total < budget)

3129.100 DT 370.900 DT True


<details><summary>💡 Indice</summary>

Chaque opérateur renvoie un **nouvel** objet `Money`. Pour accumuler : `total = total + ...`.

</details>

In [16]:
# ✅ Vérification
assert Money.from_dinars(12.5) == Money(12500) and repr(Money(12500)) == "Money(12500)"
assert str(Money(12500)) == "12.500 DT"
assert Money(1000) + Money(500) == Money(1500) and Money(1000) - Money(1500) == Money(-500)
assert Money(1500) * 3 == Money(4500) and Money(1) < Money(2)
assert total == Money(3129100) and str(remaining) == "370.900 DT"
print("✅ Exercice 7 réussi")

✅ Exercice 7 réussi


### Exercice 8 ⭐⭐ – Un pool de serveurs

Écrivez la classe `ServerPool(name: str)` qui contient des noms de serveurs (liste interne `_servers`) :
- `add(*hostnames)` : ajoute des serveurs en ignorant les doublons ;
- `__len__`, `__contains__`, `__getitem__` (accès par index) ;
- `__add__` : `pool_a + pool_b` renvoie un **nouveau** pool nommé `"a+b"` contenant les serveurs des deux (sans doublon, ordre conservé) ;
- `__str__` → `"web: 3 serveur(s)"`.

In [17]:
class ServerPool:
    """Groupe de serveurs."""

    def __init__(self, name: str) -> None:
        self.name = name
        self._servers: list[str] = []

    def add(self, *hostnames: str) -> None:
        for hostname in hostnames:
            if hostname not in self._servers:
                self._servers.append(hostname)

    def __len__(self) -> int:
        return len(self._servers)

    def __contains__(self, hostname: str) -> bool:
        return hostname in self._servers

    def __getitem__(self, index: int) -> str:
        return self._servers[index]

    def __add__(self, other: "ServerPool") -> "ServerPool":
        merged = ServerPool(f"{self.name}+{other.name}")
        merged.add(*self._servers, *other._servers)
        return merged

    def __str__(self) -> str:
        return f"{self.name}: {len(self)} serveur(s)"


web = ServerPool("web")
web.add("srv-web-01", "srv-web-02", "srv-web-01")
db = ServerPool("db")
db.add("srv-db-01", "srv-web-02")
everything = web + db
print(web, db, everything, list(everything))

web: 2 serveur(s) db: 2 serveur(s) web+db: 3 serveur(s) ['srv-web-01', 'srv-web-02', 'srv-db-01']


<details><summary>💡 Indice</summary>

Dans `__add__`, créez un nouveau `ServerPool` et réutilisez `add` avec la décompression `*`.

</details>

In [18]:
# ✅ Vérification
assert len(web) == 2 and "srv-web-02" in web and "srv-db-01" not in web
assert web[0] == "srv-web-01" and web[-1] == "srv-web-02"
assert str(everything) == "web+db: 3 serveur(s)" and list(everything) == ["srv-web-01", "srv-web-02", "srv-db-01"]
assert len(web) == 2, "web + db ne doit pas modifier web"
print("✅ Exercice 8 réussi")

✅ Exercice 8 réussi


### Exercice 9 ⭐⭐⭐ – Un onduleur avec propriétés

Écrivez la classe `UPS(name: str, capacity_va: int, charge: float = 100.0)` (onduleur) :
- propriété `charge` (en %) : le setter **ramène** la valeur dans [0, 100] (`120` → `100`, `-5` → `0`) ;
- propriété en **lecture seule** `status` : `"OK"` (≥ 50), `"LOW"` (≥ 20), `"CRITICAL"` (< 20) ;
- propriété en lecture seule `capacity_va` (stockée dans `__capacity_va`, attribut privé) ;
- méthode `discharge(minutes: int, rate: float = 1.5)` : la charge baisse de `rate` % par minute.

In [19]:
class UPS:
    """Onduleur de salle serveur."""

    def __init__(self, name: str, capacity_va: int, charge: float = 100.0) -> None:
        self.name = name
        self.__capacity_va = capacity_va
        self.charge = charge

    @property
    def capacity_va(self) -> int:
        return self.__capacity_va

    @property
    def charge(self) -> float:
        return self._charge

    @charge.setter
    def charge(self, value: float) -> None:
        self._charge = max(0.0, min(100.0, float(value)))

    @property
    def status(self) -> str:
        if self._charge >= 50:
            return "OK"
        if self._charge >= 20:
            return "LOW"
        return "CRITICAL"

    def discharge(self, minutes: int, rate: float = 1.5) -> None:
        self.charge = self.charge - minutes * rate


ups = UPS("ups-salle-a", 3000)
ups.discharge(40)
print(ups.charge, ups.status)
ups.discharge(30)
print(ups.charge, ups.status)

40.0 LOW
0.0 CRITICAL


<details><summary>💡 Indice</summary>

Dans `__init__`, écrivez `self.charge = charge` : le setter s'applique dès la création. `discharge` réutilise aussi le setter.

</details>

In [20]:
# ✅ Vérification
u = UPS("test", 1500, charge=120)
assert u.charge == 100.0 and u.status == "OK"
u.charge = -5
assert u.charge == 0.0 and u.status == "CRITICAL"
u.charge = 35
assert u.status == "LOW"
u.discharge(10, rate=2)
assert u.charge == 15.0 and u.status == "CRITICAL"
assert u.capacity_va == 1500
try:
    u.status = "OK"
    raise AssertionError("status doit être en lecture seule")
except AttributeError:
    pass
try:
    u.capacity_va = 99
    raise AssertionError("capacity_va doit être en lecture seule")
except AttributeError:
    pass
assert "_UPS__capacity_va" in vars(u)
print("✅ Exercice 9 réussi")

✅ Exercice 9 réussi


---
### Exercice de synthèse ⭐⭐⭐ – La classe IPv4Address

Écrivez la classe `IPv4Address`, qui stocke l'adresse sous forme d'un entier (attribut `value`, entre 0 et 2³² − 1) :
- `is_valid(text)` (**statique**) et `from_string(text)` (**de classe**) ;
- `__str__` → `"192.168.1.10"` ; `__repr__` → `"IPv4Address('192.168.1.10')"` ;
- `__eq__`, `__lt__` (ordre numérique, donc `10.0.0.9 < 10.0.0.10`) ;
- `__add__(n: int)` → l'adresse située `n` positions plus loin ;
- propriété `is_private` : vraie pour `10.0.0.0/8`, `172.16.0.0/12` (172.16 à 172.31) et `192.168.0.0/16`.

Utilisez-la pour trier `addresses` et générer `pool` : les 5 adresses à partir de `192.168.10.250`.

In [21]:
class IPv4Address:
    """Adresse IPv4 stockée sous forme d'entier."""

    def __init__(self, value: int) -> None:
        self.value = value

    @staticmethod
    def is_valid(text: str) -> bool:
        parts = text.split(".")
        return len(parts) == 4 and all([p.isdigit() and 0 <= int(p) <= 255 for p in parts])

    @classmethod
    def from_string(cls, text: str) -> "IPv4Address":
        a, b, c, d = [int(p) for p in text.split(".")]
        return cls(a * 256 ** 3 + b * 256 ** 2 + c * 256 + d)

    def octets(self) -> tuple[int, int, int, int]:
        v = self.value
        return v // 256 ** 3, v // 256 ** 2 % 256, v // 256 % 256, v % 256

    def __str__(self) -> str:
        return ".".join([str(o) for o in self.octets()])

    def __repr__(self) -> str:
        return f"IPv4Address('{self}')"

    def __eq__(self, other: object) -> bool:
        return isinstance(other, IPv4Address) and self.value == other.value

    def __lt__(self, other: "IPv4Address") -> bool:
        return self.value < other.value

    def __add__(self, n: int) -> "IPv4Address":
        return IPv4Address(self.value + n)

    @property
    def is_private(self) -> bool:
        a, b, _, _ = self.octets()
        return a == 10 or (a == 172 and 16 <= b <= 31) or (a == 192 and b == 168)


addresses = [IPv4Address.from_string(t) for t in ["10.0.0.10", "10.0.0.9", "192.168.1.1", "8.8.8.8"]]
print(sorted(addresses))
start = IPv4Address.from_string("192.168.10.250")
pool = [str(start + i) for i in range(5)]
print(pool, [a.is_private for a in addresses])

[IPv4Address('8.8.8.8'), IPv4Address('10.0.0.9'), IPv4Address('10.0.0.10'), IPv4Address('192.168.1.1')]
['192.168.10.250', '192.168.10.251', '192.168.10.252', '192.168.10.253', '192.168.10.254'] [True, True, True, False]


<details><summary>💡 Indice</summary>

- a.b.c.d → `a × 256³ + b × 256² + c × 256 + d`.
- Inverse : `v // 256**3`, `v // 256**2 % 256`, `v // 256 % 256`, `v % 256`.
- Pour aller plus loin, la bibliothèque standard propose le module `ipaddress`… qui fait exactement cela !

</details>

In [22]:
# ✅ Vérification
ip = IPv4Address.from_string("192.168.1.10")
assert ip.value == 3232235786 and str(ip) == "192.168.1.10" and repr(ip) == "IPv4Address('192.168.1.10')"
assert IPv4Address.is_valid("10.0.0.1") and not IPv4Address.is_valid("10.0.0.256")
assert IPv4Address.from_string("10.0.0.9") < IPv4Address.from_string("10.0.0.10")
assert IPv4Address.from_string("10.0.0.9") == IPv4Address.from_string("10.0.0.9")
assert [str(a) for a in sorted(addresses)] == ["8.8.8.8", "10.0.0.9", "10.0.0.10", "192.168.1.1"]
assert pool == ["192.168.10.250", "192.168.10.251", "192.168.10.252", "192.168.10.253", "192.168.10.254"]
assert str(IPv4Address.from_string("10.0.0.255") + 1) == "10.0.1.0"
assert [IPv4Address.from_string(t).is_private for t in ["172.20.1.1", "172.32.0.1", "8.8.8.8", "10.1.2.3"]] == [True, False, False, True]
print("✅ Exercice de synthèse réussi")

✅ Exercice de synthèse réussi


---
## Projet fil rouge – étape 5 : `Equipment` complète, `Employee` et `Inventory`

![UML étape 5](img/s07_gestparc.svg)

**À faire :**
1. **`Equipment`** :
   - attributs de classe `CATEGORIES` et `count` (nombre d'équipements créés) ;
   - `is_valid_tag(tag)` (statique) : 2 lettres, un tiret, 3 chiffres ;
   - `from_dict(data)` (de classe) et `to_dict()` ;
   - propriété `price` : le setter refuse un prix négatif (message, valeur inchangée) ;
   - propriété en lecture seule `assigned_to` (attribut interne `_assigned_to`), modifiée uniquement par `assign` / `unassign` ;
   - `__str__` (ancien `describe`), `__repr__`, `__eq__` (même numéro), `__lt__` (ordre des numéros).
2. **`Employee`** : `first_name`, `last_name`, `department`, propriété `full_name`, `__str__`.
3. **`Inventory`** : `add` (refuse catégorie inconnue, numéro invalide ou déjà présent), `find`, `assign(tag, employee)`, `stats`, `total_value`, `available`, `older_than`, `__len__`, `__contains__` (numéro **ou** équipement), `__getitem__` (index).

In [23]:
class Equipment:
    """Équipement du parc informatique."""

    CATEGORIES = {"laptop", "server", "printer", "network"}
    count = 0

    def __init__(self, asset_tag: str, category: str, brand: str, model: str,
                 purchase_year: int, price: float, assigned_to: str | None = None) -> None:
        self.asset_tag = asset_tag.upper()
        self.category = category
        self.brand = brand
        self.model = model
        self.purchase_year = purchase_year
        self._price = 0.0
        self.price = price
        self._assigned_to = assigned_to
        Equipment.count += 1

    # --- Méthodes statiques et de classe
    @staticmethod
    def is_valid_tag(tag: str) -> bool:
        """Format attendu : 2 lettres, un tiret, 3 chiffres (ex. LT-001)."""
        return len(tag) == 6 and tag[:2].isalpha() and tag[2] == "-" and tag[3:].isdigit()

    @classmethod
    def from_dict(cls, data: dict) -> "Equipment":
        """Crée un équipement à partir d'un dictionnaire (format de to_dict)."""
        return cls(data["asset_tag"], data["category"], data["brand"], data["model"],
                   data["purchase_year"], data["price"], data.get("assigned_to"))

    def to_dict(self) -> dict:
        """Représentation sous forme de dictionnaire."""
        return {"asset_tag": self.asset_tag, "category": self.category, "brand": self.brand,
                "model": self.model, "purchase_year": self.purchase_year, "price": self.price,
                "assigned_to": self.assigned_to}

    # --- Propriétés
    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if float(value) < 0:
            print(f"Prix refusé pour {self.asset_tag} : {value}")
            return
        self._price = float(value)

    @property
    def assigned_to(self) -> str | None:
        return self._assigned_to

    # --- Comportements
    def age(self, current_year: int) -> int:
        return current_year - self.purchase_year

    def is_available(self) -> bool:
        return self._assigned_to is None

    def assign(self, employee: str) -> bool:
        if not self.is_available():
            return False
        self._assigned_to = employee
        return True

    def unassign(self) -> None:
        self._assigned_to = None

    # --- Méthodes spéciales
    def __str__(self) -> str:
        name = f"{self.brand} {self.model}"
        return (f"{self.asset_tag:<7} | {self.category:<8} | {name:<25} | "
                f"{self.price:>10,.2f} DT | {self.assigned_to or '-'}")

    def __repr__(self) -> str:
        return f"Equipment({self.asset_tag!r}, {self.category!r}, {self.brand!r}, {self.model!r})"

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Equipment) and self.asset_tag == other.asset_tag

    def __lt__(self, other: "Equipment") -> bool:
        return self.asset_tag < other.asset_tag


class Employee:
    """Employé pouvant recevoir des équipements."""

    def __init__(self, first_name: str, last_name: str, department: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.department = department

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    def __str__(self) -> str:
        return f"{self.full_name} ({self.department})"


class Inventory:
    """Inventaire du parc informatique."""

    def __init__(self) -> None:
        self._items: list[Equipment] = []

    def add(self, equipment: Equipment) -> bool:
        """Ajoute un équipement valide et non présent."""
        if (equipment.category not in Equipment.CATEGORIES
                or not Equipment.is_valid_tag(equipment.asset_tag)
                or equipment in self._items):
            return False
        self._items.append(equipment)
        return True

    def find(self, asset_tag: str) -> Equipment | None:
        tag = asset_tag.upper()
        for equipment in self._items:
            if equipment.asset_tag == tag:
                return equipment
        return None

    def assign(self, asset_tag: str, employee: Employee) -> bool:
        equipment = self.find(asset_tag)
        return equipment is not None and equipment.assign(employee.full_name)

    def stats(self) -> dict[str, int]:
        result: dict[str, int] = {}
        for equipment in self._items:
            result[equipment.category] = result.get(equipment.category, 0) + 1
        return result

    def total_value(self, category: str | None = None) -> float:
        return sum([e.price for e in self._items if category is None or e.category == category])

    def available(self) -> list[Equipment]:
        return [e for e in self._items if e.is_available()]

    def older_than(self, years: int, current_year: int) -> list[Equipment]:
        return [e for e in self._items if e.age(current_year) >= years]

    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, item: object) -> bool:
        if isinstance(item, str):
            return self.find(item) is not None
        return item in self._items

    def __getitem__(self, index: int) -> Equipment:
        return self._items[index]


inventory = Inventory()
inventory.add(Equipment("SV-001", "server", "HPE", "ProLiant DL380", 2019, 18500))
inventory.add(Equipment("LT-001", "laptop", "Dell", "Latitude 5440", 2023, 3200))
inventory.add(Equipment.from_dict({"asset_tag": "PR-001", "category": "printer", "brand": "HP",
                                   "model": "LaserJet M404", "purchase_year": 2021, "price": 950}))
amira = Employee("Amira", "Ben Salah", "DSI")
inventory.assign("lt-001", amira)
for equipment in sorted(inventory):
    print(equipment)
print(len(inventory), "LT-001" in inventory, inventory.stats(), f"{inventory.total_value():,.2f} DT")

LT-001  | laptop   | Dell Latitude 5440        |   3,200.00 DT | Amira Ben Salah
PR-001  | printer  | HP LaserJet M404          |     950.00 DT | -
SV-001  | server   | HPE ProLiant DL380        |  18,500.00 DT | -
3 True {'server': 1, 'laptop': 1, 'printer': 1} 22,650.00 DT


In [24]:
# ✅ Vérification
before = Equipment.count
e = Equipment("lt-050", "laptop", "Asus", "ExpertBook", 2024, 2500)
assert Equipment.count == before + 1
assert Equipment.is_valid_tag("LT-050") and not Equipment.is_valid_tag("LT-50") and not Equipment.is_valid_tag("L1-050")
e.price = -10
assert e.price == 2500.0
try:
    e.assigned_to = "X"
    raise AssertionError("assigned_to doit être en lecture seule")
except AttributeError:
    pass
assert Equipment.from_dict(e.to_dict()) == e
assert repr(e) == "Equipment('LT-050', 'laptop', 'Asus', 'ExpertBook')"
assert str(e) == "LT-050  | laptop   | Asus ExpertBook           |   2,500.00 DT | -"
assert Employee("Karim", "Trabelsi", "Finance").full_name == "Karim Trabelsi"
inv = Inventory()
assert inv.add(e) and not inv.add(Equipment("LT-050", "laptop", "X", "Y", 2024, 1))
assert not inv.add(Equipment("TB-001", "tablet", "X", "Y", 2024, 1)) and not inv.add(Equipment("LAPTOP-1", "laptop", "X", "Y", 2024, 1))
assert len(inventory) == 3 and "pr-001" in inventory and inventory[0].asset_tag == "SV-001"
assert inventory.find("LT-001").assigned_to == "Amira Ben Salah"
assert not inventory.assign("LT-001", Employee("Karim", "Trabelsi", "Finance"))
assert [x.asset_tag for x in sorted(inventory)] == ["LT-001", "PR-001", "SV-001"]
assert [x.asset_tag for x in inventory.available()] == ["SV-001", "PR-001"]
assert inventory.total_value() == 22650.0 and inventory.stats() == {"server": 1, "laptop": 1, "printer": 1}
assert [x.asset_tag for x in inventory.older_than(5, 2026)] == ["SV-001", "PR-001"]
print("✅ Étape 5 du projet GestParc validée")

Prix refusé pour LT-050 : -10
✅ Étape 5 du projet GestParc validée


✔️ Version de référence : `scripts/gestparc/main.py`.

**Prochaine étape (S9)** : après le TP de S8, `Equipment` deviendra une classe abstraite, avec les sous-classes `Laptop`, `Server`, `Printer` et `NetworkDevice`.

---
## Pour aller plus loin (bonus)

1. **`functools.total_ordering`** : décorez `SoftwareVersion` et vérifiez que `<=`, `>` et `>=` fonctionnent avec seulement `__eq__` et `__lt__`.
2. **`__hash__`** : essayez `{SoftwareVersion("3.12")}`. Pourquoi Python refuse-t-il ? Ajoutez `__hash__` (`return hash(self.parts)`).
3. **`__iter__`** : remplacez le `__getitem__` d'`Inventory` par `def __iter__(self): return iter(self._items)` ; qu'est-ce qui fonctionne encore ?
4. Comparez votre `IPv4Address` avec le module standard `ipaddress` : `ipaddress.ip_address("192.168.1.10").is_private`.

In [25]:
import ipaddress
from functools import total_ordering

SoftwareVersion = total_ordering(SoftwareVersion)
print(SoftwareVersion("3.12") >= SoftwareVersion("3.9"))
print(ipaddress.ip_address("192.168.1.10").is_private, int(ipaddress.ip_address("192.168.1.10")))

True
True 3232235786
